In [9]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd 
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from google.colab import runtime
runtime.unassign()
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Colab drive not available; using local paths if present.")

Colab drive not available; using local paths if present.


In [2]:

root_candidates = [
    Path.cwd() / "data" / "mallorn-astronomical-classification-challenge",
    Path("/content/drive/MyDrive/data/mallorn-astronomical-classification-challenge/"),
    Path("c:/Users/mlady/Documents/202620/stat6685/finalproject/data/mallorn-astronomical-classification-challenge"),
]
root = next((path for path in root_candidates if path.exists()), root_candidates[0])
# 1) Global training metadata with labels
train_log = pd.read_csv(root / "train_log.csv")
test_log = pd.read_csv(root / "test_log.csv")

# 2) Load one split's lightcurves and attach labels/features from global log
def load_split(split_name: str, is_test=False):
    if is_test:
        lc_path = root / split_name / "test_full_lightcurves.csv"
    else:
        lc_path = root / split_name / "train_full_lightcurves.csv"
    lc = pd.read_csv(lc_path)

    log = test_log if is_test else train_log

    # Keep only metadata rows for this split
    meta = log.loc[log["split"] == split_name].copy()

    # Merge target + static features onto each observation row
    merge_cols = ["object_id", "Z", "Z_err", "EBV", "SpecType", "split"]
    if not is_test and "target" in meta.columns:
        merge_cols.insert(1, "target")
    merge_cols = [col for col in merge_cols if col in meta.columns]
    merged = lc.merge(
        meta[merge_cols],
        on="object_id",
        how="left",
        validate="many_to_one"
    )
    return merged, meta

In [3]:
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTER_ORDER)}

def encode_sequence(obj_df):
    obj_df = obj_df.sort_values("Time (MJD)").copy()

    # Robust numeric conversion + NaN handling to prevent NaN loss during training
    t = pd.to_numeric(obj_df["Time (MJD)"], errors="coerce").to_numpy(dtype=np.float32)
    flux = pd.to_numeric(obj_df["Flux"], errors="coerce").to_numpy(dtype=np.float32)
    flux_err = pd.to_numeric(obj_df["Flux_err"], errors="coerce").to_numpy(dtype=np.float32)

    t = np.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
    flux = np.nan_to_num(flux, nan=0.0, posinf=0.0, neginf=0.0)
    flux_err = np.nan_to_num(flux_err, nan=0.0, posinf=0.0, neginf=0.0)

    dt = np.diff(t, prepend=t[0]).astype(np.float32)
    dt = np.clip(dt, a_min=0.0, a_max=1e4)

    # Stabilize dynamic ranges for RNN training
    flux = np.clip(flux, -1e5, 1e5)
    flux_err = np.clip(flux_err, 0.0, 1e5)

    filt_oh = np.zeros((len(obj_df), len(FILTER_ORDER)), dtype=np.float32)
    filt_idx = obj_df["Filter"].map(FILTER_TO_IDX).to_numpy()
    valid = ~pd.isna(filt_idx)
    filt_oh[np.arange(len(obj_df))[valid], filt_idx[valid].astype(int)] = 1.0

    x = np.column_stack([flux, flux_err, dt, filt_oh]).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return torch.tensor(x, dtype=torch.float32)

class LightCurveSplitDataset(Dataset):
    def __init__(self, split_name, root_path, is_test=False):
        self.split_name = split_name
        self.root_path = root_path
        self.is_test = is_test

        self.merged, self.meta = load_split(split_name, is_test=is_test)

        self.meta = self.meta.set_index("object_id", drop=False)

        object_set = set(self.merged["object_id"].unique())
        self.object_ids = [oid for oid in self.meta["object_id"].tolist() if oid in object_set]
        self.grouped = dict(tuple(self.merged.groupby("object_id")))

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        oid = self.object_ids[idx]
        obj_df = self.grouped[oid]
        m = self.meta.loc[oid]

        seq = encode_sequence(obj_df)

        z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
        z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
        ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
        static_np = np.array([z, z_err, ebv], dtype=np.float32)
        static_np = np.nan_to_num(static_np, nan=0.0, posinf=0.0, neginf=0.0)
        static_np = np.clip(static_np, -1e3, 1e3)
        static = torch.tensor(static_np, dtype=torch.float32)

        sample = {
            "object_id": oid,
            "seq": seq,
            "static": static,
        }

        if not self.is_test and "target" in self.merged.columns:
            sample["y"] = torch.tensor(float(m["target"]), dtype=torch.float32)

        return sample

"""
Build one mini-batch from a list of dataset samples.
"""
def collate_pad(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
        "y": y
    }

def collate_pad_test(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
    }

# Example usage:
train_ds = LightCurveSplitDataset("split_01", root)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)

batch = next(iter(train_loader))

In [4]:
class LSTMWithStatic(nn.Module):
    def __init__(self, seq_input_size=9, static_input_size=3, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.static_net = nn.Sequential(
            nn.Linear(static_input_size, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size + 16, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, seq, lengths, static):
        packed = nn.utils.rnn.pack_padded_sequence(
            seq, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        seq_repr = h_n[-1]
        static_repr = self.static_net(static)
        x = torch.cat([seq_repr, static_repr], dim=1)
        logits = self.head(x).squeeze(1)
        return logits

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMWithStatic(seq_input_size=9, static_input_size=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

all_splits = sorted(train_log["split"].dropna().unique().tolist())
val_splits = [all_splits[-1]]  # hold out last split for validation
train_splits = [s for s in all_splits if s not in val_splits]

print("Train splits:", train_splits)
print("Val splits:", val_splits)

# Build loaders
train_loaders = {}
val_loaders = {}
all_targets = []

for split_name in train_splits:
    ds = LightCurveSplitDataset(split_name, root)
    train_loaders[split_name] = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_pad)
    all_targets.append(ds.meta["target"].astype(float).values)

for split_name in val_splits:
    ds = LightCurveSplitDataset(split_name, root)
    val_loaders[split_name] = DataLoader(ds, batch_size=128, shuffle=False, collate_fn=collate_pad)

all_targets = np.concatenate(all_targets)
pos = float((all_targets == 1).sum())
neg = float((all_targets == 0).sum())
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Class balance (train only) -> pos: {int(pos)}, neg: {int(neg)}, pos_weight: {pos_weight.item():.2f}")

def eval_on_loaders(loaders, threshold=0.5):
    model.eval()
    logits_all = []
    y_all = []
    with torch.no_grad():
        for split_name in loaders:
            for batch in loaders[split_name]:
                seq = batch["seq"].to(device)
                lengths = batch["lengths"]
                static = batch["static"].to(device)
                y = batch["y"].to(device)

                logits = model(seq, lengths, static)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                logits_all.append(logits.cpu())
                y_all.append(y.cpu())

    logits_all = torch.cat(logits_all)
    y_all = torch.cat(y_all)
    probs = torch.sigmoid(logits_all)
    preds = (probs >= threshold).float()

    acc = (preds == y_all).float().mean().item()
    precision = ((preds * y_all).sum() / (preds.sum() + 1e-8)).item()
    recall = ((preds * y_all).sum() / (y_all.sum() + 1e-8)).item()
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)
    return {
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "probs": probs,
        "y": y_all,
    }

def tune_threshold(probs, y_true):
    best_thr = 0.5
    best_f1 = -1.0
    for thr in np.arange(0.05, 0.96, 0.01):
        preds = (probs >= thr).float()
        precision = ((preds * y_true).sum() / (preds.sum() + 1e-8)).item()
        recall = ((preds * y_true).sum() / (y_true.sum() + 1e-8)).item()
        f1 = (2 * precision * recall) / (precision + recall + 1e-8)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1

n_epochs = 100
patience = 12
epochs_without_improve = 0
best_val_f1 = -1.0
best_threshold = 0.5
best_state_dict = None

for epoch in range(1, n_epochs + 1):
    model.train()
    running_loss = 0.0
    n_obs = 0

    for split_name in train_splits:
        loader = train_loaders[split_name]
        for batch in loader:
            seq = batch["seq"].to(device)
            lengths = batch["lengths"]
            static = batch["static"].to(device)
            y = batch["y"].to(device)

            logits = model(seq, lengths, static)
            loss = loss_fn(logits, y)

            if torch.isnan(loss):
                print(f"NaN loss detected in {split_name}. Skipping this batch.")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = y.size(0)
            running_loss += loss.item() * bs
            n_obs += bs

    epoch_loss = running_loss / max(n_obs, 1)

    if epoch == 1 or epoch % 2 == 0:
        train_metrics = eval_on_loaders(train_loaders, threshold=0.5)
        val_metrics_05 = eval_on_loaders(val_loaders, threshold=0.5)
        tuned_thr, tuned_val_f1 = tune_threshold(val_metrics_05["probs"], val_metrics_05["y"])
        val_metrics_tuned = eval_on_loaders(val_loaders, threshold=tuned_thr)

        print(
            f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | "
            f"train_f1@0.50={train_metrics['f1']:.4f} | "
            f"val_f1@0.50={val_metrics_05['f1']:.4f} | "
            f"val_f1@{tuned_thr:.2f}={val_metrics_tuned['f1']:.4f}"
        )

        if val_metrics_tuned["f1"] > best_val_f1 + 1e-6:
            best_val_f1 = val_metrics_tuned["f1"]
            best_threshold = tuned_thr
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        if epochs_without_improve >= patience:
            print(f"Early stopping at epoch {epoch} (best val_f1={best_val_f1:.4f}, best_threshold={best_threshold:.2f})")
            break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Loaded best checkpoint with val_f1={best_val_f1:.4f} at threshold={best_threshold:.2f}")
else:
    best_threshold = 0.5
    print("No validation improvement tracked; using threshold=0.5")

Train splits: ['split_01', 'split_02', 'split_03', 'split_04', 'split_05', 'split_06', 'split_07', 'split_08', 'split_09', 'split_10', 'split_11', 'split_12', 'split_13', 'split_14', 'split_15', 'split_16', 'split_17', 'split_18', 'split_19']
Val splits: ['split_20']
Class balance (train only) -> pos: 139, neg: 2751, pos_weight: 19.79
Epoch 01 | loss=1.3246 | train_f1@0.50=0.1016 | val_f1@0.50=0.1406 | val_f1@0.50=0.1406
Epoch 02 | loss=1.3176 | train_f1@0.50=0.1125 | val_f1@0.50=0.1633 | val_f1@0.51=0.1647
Epoch 04 | loss=1.2703 | train_f1@0.50=0.1075 | val_f1@0.50=0.0392 | val_f1@0.22=0.1391
Epoch 06 | loss=1.3022 | train_f1@0.50=0.1265 | val_f1@0.50=0.0526 | val_f1@0.36=0.1739
Epoch 08 | loss=1.2552 | train_f1@0.50=0.1565 | val_f1@0.50=0.2273 | val_f1@0.56=0.2857
Epoch 10 | loss=1.2293 | train_f1@0.50=0.1779 | val_f1@0.50=0.1356 | val_f1@0.40=0.2051
Epoch 12 | loss=1.1780 | train_f1@0.50=0.1725 | val_f1@0.50=0.3000 | val_f1@0.59=0.3077
Epoch 14 | loss=1.1948 | train_f1@0.50=0.2028 |

In [6]:
# Save trained model (and training state) to Google Drive
ckpt_dir = root / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Resolve missing references safely
eval_threshold = float(best_threshold) if "best_threshold" in globals() else 0.5

if "val_loaders" in globals() and len(val_loaders) > 0:
    metrics = eval_on_loaders(val_loaders, threshold=eval_threshold)
elif "train_loaders" in globals() and len(train_loaders) > 0:
    metrics = eval_on_loaders(train_loaders, threshold=eval_threshold)
else:
    metrics = {"acc": None, "precision": None, "recall": None, "f1": None}

ckpt_path = ckpt_dir / "lstm_with_static_split01_20_apr16.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": int(epoch) if "epoch" in globals() else None,
        "loss": float(epoch_loss) if "epoch_loss" in globals() else None,
        "acc": metrics["acc"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "best_val_f1": float(best_val_f1) if "best_val_f1" in globals() else None,
        "best_threshold": eval_threshold,
        "pos_weight": pos_weight.detach().cpu() if "pos_weight" in globals() else None,
        "split_names": {
            "train_splits": train_splits if "train_splits" in globals() else [],
            "val_splits": val_splits if "val_splits" in globals() else [],
        },
    },
    ckpt_path,
)

print(f"Saved checkpoint to: {ckpt_path}")

Saved checkpoint to: /content/drive/MyDrive/data/mallorn-astronomical-classification-challenge/checkpoints/lstm_with_static_split01_20_apr16.pt


In [8]:
model.eval()

test_split_names = sorted(test_log["split"].dropna().unique().tolist())
pred_rows = []
batch_size = 128
inference_threshold = float(best_threshold) if "best_threshold" in globals() else 0.5
print(f"Using inference threshold: {inference_threshold:.2f}")

with torch.no_grad():
    for split_name in test_split_names:
        lc = pd.read_csv(root / split_name / "test_full_lightcurves.csv")
        meta = test_log.loc[test_log["split"] == split_name].copy().set_index("object_id", drop=False)
        grouped = dict(tuple(lc.groupby("object_id")))
        object_ids = [oid for oid in meta["object_id"].tolist() if oid in grouped]

        for start in range(0, len(object_ids), batch_size):
            batch_ids = object_ids[start:start + batch_size]
            seqs = []
            statics = []

            for object_id in batch_ids:
                obj_df = grouped[object_id]
                seqs.append(encode_sequence(obj_df))

                m = meta.loc[object_id]
                z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
                z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
                ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
                static_np = np.array([z, z_err, ebv], dtype=np.float32)
                static_np = np.nan_to_num(static_np, nan=0.0, posinf=0.0, neginf=0.0)
                statics.append(torch.tensor(np.clip(static_np, -1e3, 1e3), dtype=torch.float32))

            lengths = torch.tensor([seq.size(0) for seq in seqs], dtype=torch.long)
            seq_padded = pad_sequence(seqs, batch_first=True)
            static = torch.stack(statics, dim=0)

            logits = model(seq_padded.to(device), lengths, static.to(device))
            preds = (torch.sigmoid(logits).cpu().numpy() >= inference_threshold).astype(int)

            for object_id, pred in zip(batch_ids, preds):
                pred_rows.append({"object_id": object_id, "prediction": int(pred)})

pred_df = pd.DataFrame(pred_rows)
sample_submission = pd.read_csv(root / "sample_submission.csv")
pred_df = sample_submission[["object_id"]].merge(pred_df, on="object_id", how="left")
pred_df["prediction"] = pred_df["prediction"].fillna(0).astype(int)

output_path = root / "submission.csv"
pred_df.to_csv(output_path, index=False)
print(f"Saved predictions to: {output_path}")
print(pred_df.head(12).to_csv(index=False).strip())

Using inference threshold: 0.47
Saved predictions to: /content/drive/MyDrive/data/mallorn-astronomical-classification-challenge/submission.csv
object_id,prediction
Eluwaith_Mithrim_nothrim,0
Eru_heledir_archam,0
Gonhir_anann_fuin,0
Gwathuirim_haradrim_tegilbor,1
achas_minai_maen,0
adab_fae_gath,0
adel_draug_gaur,0
aderthad_cuil_galadhrim,0
aegas_laug_ithildin,0
aegas_mereth_law,0
agar_heron_salph,0
agarwaen_glae_ithildin,0
